In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import GridSearchCV
import pandas as pd
import joblib
import numpy as np
import sklearn

sklearn.set_config(transform_output="pandas")
pd.set_option('display.max_columns', None)

transformer = joblib.load("transformer.pkl")
train_df    = pd.read_csv("data/balanced_train_data.csv")

y        = train_df['TARGET'].astype(np.float32)
train_df = train_df.drop(columns=['TARGET'])

X_transformed = transformer.transform(train_df).astype(np.float32)

del train_df

param_grid = {
    'n_estimators': [100],
    'max_depth': [None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

rclr = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
)
cv = GridSearchCV(estimator=rclr, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2)

cv.fit(X=X_transformed, y=y)

test_df = pd.read_csv("data/split_data_test.csv")
y_test = test_df['TARGET'].astype(np.float32)
X_test = test_df.drop(columns=['TARGET'])
transformed_test_df = transformer.transform(X_test).astype(np.float32)

del X_test

y_probs = cv.predict_proba(X=transformed_test_df)[:, 1]
y_pred  = cv.predict(X=transformed_test_df)

result = pd.DataFrame({
    'library': ["scikit-learn"],
    'algorithm': ['RandomForestClassifier + GridSearchCV'],
    'hyerparamters':[str(cv.best_params_)],
    'Accuracy': [accuracy_score(y_test, y_pred)],
    'AUC': [roc_auc_score(y_test, y_probs)]
}, index=None)

print(result.T)


Fitting 3 folds for each of 4 candidates, totalling 12 fits
                                                               0
library                                             scikit-learn
algorithm                  RandomForestClassifier + GridSearchCV
hyerparamters  {'max_depth': None, 'min_samples_leaf': 1, 'mi...
Accuracy                                                0.917481
AUC                                                       0.8191
